# Notebook 3: Validate Exported Adapter Directly

This notebook loads one adapter directly, reads its metadata, predicts a single image and, optionally, runs a quick health check over a folder.
The single-image prediction can also show deterministic derived-view results from the same file, an optional occlusion sensitivity map or a DINOv3 ViT attention map.

The notebook searches the project folder and common Drive roots for adapter bundles. A path can also be given by hand.

Manual inputs:
- `ADAPTER_DIR`: adapter folder, export folder, telemetry run folder or `adapter_meta.json`
- `ADAPTER_ROOT`: parent of the crop folders, such as `models/adapters/`
- `IMAGE_PATH`: a single image
- `BATCH_IMAGE_DIR`: a folder of images

To try again in the same session:
- set `FORCE_ADAPTER_RESCAN = True` and re-run the adapter discovery cell
- the single-image cell asks for a new upload by default
- set `REUSE_EXISTING_IMAGE_PATH = True` to keep the current upload
- set `FORCE_IMAGE_UPLOAD = True` to force a new upload right away


## Quick Flow

This notebook skips the router and runs a smoke test and single-image prediction directly on the selected adapter bundle.

- If no adapter is found automatically, set `ADAPTER_DIR`; otherwise pick `SELECTED_ADAPTER_INDEX` from the list.
- Use `FORCE_IMAGE_UPLOAD = True` for a new image and `REUSE_EXISTING_IMAGE_PATH = True` to keep the same image.
- Scores are a quick health check; do not read them as calibrated accuracy probabilities.
- For the full router-plus-adapter prediction, use Notebook 8.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/aads-open-world-plant-disease.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if CLONE_TARGET.exists():
        shutil.rmtree(CLONE_TARGET)
    print(f'[BOOTSTRAP] Cloning repo to {CLONE_TARGET}...', flush=True)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell01_bootstrap_access.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell02_access_notice.py', globals())


In [ ]:
import json
from pathlib import Path

from IPython.display import display, HTML

# Quick use: pick SELECTED_ADAPTER_INDEX if an adapter is found automatically; otherwise set ADAPTER_DIR.
# IMAGE_PATH=None uses the upload flow in Colab.
CROP_NAME = None
# Example: 'tomato'. Leave None to list all adapters.

ADAPTER_DIR = None
# Accepted examples:
# - .../continual_sd_lora_adapter/
# - .../artifacts/adapter_export/
# - outputs/colab_notebook_training/
# - .../telemetry/<RUN_ID>/ or .../telemetry/<RUN_ID>/artifacts/
# - .../adapter_meta.json

ADAPTER_ROOT = None
# Usually the parent of crop folders like models/adapters/

INCLUDE_RUN_ADAPTERS = False
# Set True only when you intentionally want to inspect historical runs/ artifacts.

SEARCH_ROOTS = [
    ROOT / 'outputs' / 'colab_notebook_training',
    ROOT / 'outputs' / 'colab_notebook_training' / 'telemetry_runtime' / 'telemetry',
    ROOT / 'models' / 'adapters',
]
if INCLUDE_RUN_ADAPTERS:
    SEARCH_ROOTS.append(ROOT / 'runs')
SELECTED_ADAPTER_INDEX = 0
FORCE_ADAPTER_RESCAN = False
IMAGE_PATH = None
FORCE_IMAGE_UPLOAD = False
# Single image file
BATCH_IMAGE_DIR = None
# Optional folder for batch testing
DEVICE = 'cuda'
CONFIG_ENV = 'colab'
REUSE_EXISTING_IMAGE_PATH = False
ENABLE_ROBUST_SMOKE = True
ROBUST_VIEWS = ('full_resize', 'resize_pad', 'center_crop')
ENABLE_PREDICTION_VISUALIZATION = True
EXPLANATION_METHOD = 'attention_map'
# Options: 'attention_map' or 'occlusion_sensitivity'
EXPLANATION_GRID_SIZE = 7

print(f'[SETTINGS] device={DEVICE} config_env={CONFIG_ENV}')
print('[ADAPTER] If ADAPTER_DIR is set, discovery is skipped; otherwise adapters are searched under SEARCH_ROOTS.')
print('[ADAPTER] Search roots:')
for root in SEARCH_ROOTS:
    print(f'- {root}')
print('[PATHS] ADAPTER_ROOT is the parent of the crop folders, IMAGE_PATH a single file, BATCH_IMAGE_DIR a folder.')
print('[RESCAN] Set FORCE_ADAPTER_RESCAN=True to rebuild the adapter list.')
print('[IMAGE] The single-image cell asks for a new upload by default; set REUSE_EXISTING_IMAGE_PATH=True to keep the current image.')
print(f'[SMOKE] robust={ENABLE_ROBUST_SMOKE} views={ROBUST_VIEWS}')
print(f'[EXPLAIN] enabled={ENABLE_PREDICTION_VISUALIZATION} method={EXPLANATION_METHOD} grid={EXPLANATION_GRID_SIZE}x{EXPLANATION_GRID_SIZE}')
print('[NEXT] Run the adapter discovery cell, then the adapter summary and single-image cells.')


In [ ]:
# Defer adapter discovery imports until now (when actually needed)
try:
    import ipywidgets as widgets
except Exception:
    widgets = None

from src.colab.adapter_smoke import discover_adapter_candidates

adapter_candidates = []
adapter_selector = None

if FORCE_ADAPTER_RESCAN:
    ADAPTER_DIR = None
    print('FORCE_ADAPTER_RESCAN=True, rebuilding adapter list.')

if ADAPTER_DIR is not None:
    SELECTED_ADAPTER_DIR = str(Path(ADAPTER_DIR))
    print(f'Using provided ADAPTER_DIR: {SELECTED_ADAPTER_DIR}')
else:
    adapter_candidates = discover_adapter_candidates(SEARCH_ROOTS, crop_name=CROP_NAME)
    if not adapter_candidates:
        raise FileNotFoundError(
            'No adapters found under SEARCH_ROOTS. If needed, provide ADAPTER_DIR manually or update SEARCH_ROOTS.'
        )

    print('Found adapters:')
    for index, candidate in enumerate(adapter_candidates):
        print(f'[{index}] {candidate["display_name"]}')

    if widgets is not None:
        adapter_selector = widgets.Dropdown(
            options=[(candidate['display_name'], index) for index, candidate in enumerate(adapter_candidates)],
            value=min(SELECTED_ADAPTER_INDEX, len(adapter_candidates) - 1),
            description='Adapter:',
            layout=widgets.Layout(width='95%'),
        )
        display(adapter_selector)
        print('Select adapter from dropdown, then run next cell.')
    else:
        print('ipywidgets not available. Set SELECTED_ADAPTER_INDEX to your desired adapter and run next cell.')

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell05_adapter_summary.py', globals())


In [ ]:
# Defer prediction and visualization imports until needed (now)
try:
    import pandas as pd
except Exception:
    pd = None

from src.colab.adapter_smoke import (
    build_prediction_visualization_images,
    predict_single_image,
)

request_new_upload = running_in_colab() and not bool(REUSE_EXISTING_IMAGE_PATH)
adapter_summary = globals().get('summary')
if (ADAPTER_DIR is None or CROP_NAME is None) and isinstance(adapter_summary, dict):
    ADAPTER_DIR = adapter_summary.get('resolved_adapter_dir', ADAPTER_DIR)
    CROP_NAME = adapter_summary.get('crop_name', CROP_NAME)

if ADAPTER_DIR is None and CROP_NAME is None:
    raise RuntimeError(
        'Run adapter selection/summary cell first so Notebook 3 can resolve ADAPTER_DIR and CROP_NAME.'
    )

if FORCE_IMAGE_UPLOAD:
    IMAGE_PATH = None
    print('FORCE_IMAGE_UPLOAD=True, requesting new image upload.')
elif request_new_upload:
    IMAGE_PATH = None
    print('REUSE_EXISTING_IMAGE_PATH=False, new image upload requested for this run.')

if IMAGE_PATH is None:
    if not running_in_colab():
        raise ValueError('Running outside Colab: set IMAGE_PATH to a single image file.')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError('Upload cancelled or no file selected.')
    IMAGE_PATH = next(iter(uploaded.keys()))
    FORCE_IMAGE_UPLOAD = False

IMAGE_PATH = str(Path(IMAGE_PATH))
print(f'Using IMAGE_PATH: {IMAGE_PATH}')

single_result = predict_single_image(
    IMAGE_PATH,
    CROP_NAME,
    adapter_dir=ADAPTER_DIR,
    adapter_root=ADAPTER_ROOT,
    config_env=CONFIG_ENV,
    device=DEVICE,
    enable_robust_smoke=ENABLE_ROBUST_SMOKE,
    robust_views=ROBUST_VIEWS,
    explain_prediction=ENABLE_PREDICTION_VISUALIZATION,
    explanation_grid_size=EXPLANATION_GRID_SIZE,
    explanation_method=EXPLANATION_METHOD,
)

view_consistency = dict(single_result.get('view_consistency', {}))
uncertainty = dict(single_result.get('uncertainty_diagnostics', {}))

print('Notebook 3 note: class confidence is top-1 softmax score, not calibrated accuracy probability.')
print('OOD decision and class confidence answer different questions; OOD summarizes distribution fit, confidence summarizes class preference.')

summary_rows = [
    {'metric': 'status', 'value': single_result.get('status')},
    {'metric': 'error', 'value': single_result.get('error')},
    {'metric': 'predicted_class', 'value': single_result.get('predicted_class')},
    {'metric': 'confidence', 'value': None if single_result.get('status') == 'error' else round(float(single_result.get('confidence', 0.0)), 4)},
    {'metric': 'is_ood', 'value': single_result.get('is_ood')},
    {'metric': 'score_method', 'value': single_result.get('score_method')},
    {'metric': 'primary_score', 'value': single_result.get('primary_score')},
    {'metric': 'decision_threshold', 'value': single_result.get('decision_threshold')},
    {'metric': 'primary_view', 'value': view_consistency.get('primary_view')},
    {'metric': 'stable_across_views', 'value': view_consistency.get('stable')},
]
if pd is not None:
    display(pd.DataFrame(summary_rows))
else:
    print(summary_rows)

view_rows = list(single_result.get('views', []))
if view_rows:
    print('Derived view results:')
    if pd is not None:
        display(pd.DataFrame(view_rows))
    else:
        print(view_rows)

visualization_images = build_prediction_visualization_images(IMAGE_PATH, single_result)
if visualization_images:
    print('Model view and explanation heatmap. Attention map shows attention routing; occlusion sensitivity shows confidence drop when bright regions are masked.')
    display(visualization_images['model_view'])
    display(visualization_images['heatmap_overlay'])

print('Prediction status:', single_result.get('status'))
if single_result.get('error'):
    print('Prediction error:', single_result.get('error'))
print('View warnings:', view_consistency.get('warning_codes', []))
print('Uncertainty warnings:', uncertainty.get('warning_codes', []))
display(
    HTML(
        '<details><summary>Raw JSON</summary><pre>'
        + json.dumps(single_result, indent=2)
        + '</pre></details>'
    )
)

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb3_cell07_batch_prediction.py', globals())
